In [1]:
import ee
import os
import pandas as pd
from config import *

# ee.Authenticate()
ee.Initialize()

c:\Users\Rawan\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
c:\Users\Rawan\Desktop\GSoC26\config.py:118: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [2]:
def _get_region_df(info):
    """Convert GEE getRegion output to DataFrame"""
    col_dict = {k: i + 4 for i, k in enumerate(info[0][4:])}
    data = {k: {} for k in col_dict}

    for row in info[1:]:
        date = pd.Timestamp(row[3] * 1_000_000)
        for band, idx in col_dict.items():
            data[band][date] = row[idx]

    return pd.DataFrame.from_dict(data)

# --------------------------------------------------------------
def download_era5_timeseries(lat, 
                             lon, 
                             start_date, 
                             end_date, 
                             bands, 
                             output_csv= "dataframes/df_soil_moisture.csv"):
    
    geometry = ee.Geometry.Point([lon, lat])
    
    collection = (
        ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR")
        .select(bands)
        .filterBounds(geometry)
        .filterDate(start_date, end_date)
    )

    if collection.size().getInfo() == 0:
        print("No ERA5 data found.")
        return None

    info = collection.getRegion(
        geometry,
        crs="EPSG:32636",
        scale=10
    ).getInfo()

    df = _get_region_df(info)
    df.to_csv(output_csv)
    print(f"ERA5 saved: {output_csv}")

    return df
# --------------------------------------------------------------


df = download_era5_timeseries(fields_location[0], fields_location[1], start_date_str, end_date_str, era5_bands)


ERA5 saved: dataframes/df_soil_moisture.csv
